# Live MSCoRe with deadlock recovery · v3.2.0
Select a GPU runtime. Run cells 1–5 in order. Three tasks provide a pilot. The 20-task selection includes the same three. Each question is presented in its original Chinese to three personas using one local model. The board records revisions, challenges and votes. Select RECOVERY_ENABLED in cell 2 for a paired comparison. Each branch allows an analyst rebuttal and a second review round before declaring a deadlock. With recovery enabled, on a detected deadlock the panel receives its prior arguments and reconsiders the original statement on one fresh branch. Run cell 6 for task-switchable recorded playback. Exact-match grading uses the released yes/no labels after discussion.

Source: [MSCoRe by 032564yn](https://huggingface.co/datasets/032564yn/MSCoRe), Construction/simple judgments. This fixed prefix is a subset evaluation.
Consensus measures agreement on the yes/no label. Agents cast structured ballots and can record caveats separately. First ballots on the original branch are independent. Subsequent votes and recovery use the discussion.

In [ ]:
#@title 1. Install the pinned project revision
NOTEBOOK_VERSION = "3.2.0"
print("MSCoRe notebook v" + NOTEBOOK_VERSION)
readiness = None
import importlib
import importlib.metadata as metadata
import os
import re
import subprocess
import sys
from tempfile import TemporaryDirectory
from pathlib import Path

PROJECT_REVISION = "ccb8afedd95d2bee040d39368e785b76fbbf68ca" #@param {type:"string"}
CHECKOUT_ROOT = Path("/content/intelligible-blackboard-checkouts")
LEGACY_SOURCE = Path("/content/intelligible-blackboard/src").resolve()
REPO_URL = "https://github.com/nikhil0744/intelligible-blackboard-architecture.git"
if not re.fullmatch(r"[0-9a-f]{40}", PROJECT_REVISION):
    raise ValueError("Use an immutable 40-character project commit, not a branch name.")
PROJECT_DIR = CHECKOUT_ROOT / PROJECT_REVISION
CHECKOUT_ROOT.mkdir(parents=True, exist_ok=True)
if not PROJECT_DIR.exists():
    with TemporaryDirectory(prefix="checkout-", dir=CHECKOUT_ROOT) as staging:
        checkout = Path(staging) / "project"
        subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(checkout)], check=True)
        subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", PROJECT_REVISION], check=True)
        checkout.rename(PROJECT_DIR)
actual_revision = subprocess.check_output(["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"], text=True).strip()
if actual_revision != PROJECT_REVISION:
    raise ValueError(f"The checkout at {PROJECT_DIR} has unexpected revision {actual_revision}. Use an intact checkout.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_DIR) + "[colab]"], check=True)
os.chdir(PROJECT_DIR)
for name in ("numpy", "pandas", "pydantic"):
    loaded = sys.modules.get(name)
    if loaded is not None and getattr(loaded, "__version__", None) != metadata.version(name):
        raise RuntimeError("A loaded package changed during installation. Restart the Colab session, then rerun the notebook.")
# Swap only source modules belonging to earlier notebook checkouts.
source_dir = (PROJECT_DIR / "src").resolve()
old_sources = {LEGACY_SOURCE}
old_sources.update(path.resolve() / "src" for path in CHECKOUT_ROOT.iterdir()
                   if path != PROJECT_DIR and re.fullmatch(r"[0-9a-f]{40}", path.name))
def from_previous_checkout(path):
    return bool(path) and any(Path(path).resolve().is_relative_to(old) for old in old_sources)

stale = [name for name, module in tuple(sys.modules.items())
         if from_previous_checkout(getattr(module, "__file__", None))]
if stale and "broker" in globals():
    broker.close()
for name in stale:
    del sys.modules[name]
sys.path[:] = [str(source_dir)] + [path for path in sys.path
                                  if path != str(source_dir) and not from_previous_checkout(path)]
importlib.invalidate_caches()
from benchmarks import batch as benchmark_batch
if not Path(benchmark_batch.__file__).resolve().is_relative_to(source_dir):
    raise RuntimeError("This session loaded a different benchmark package. Restart the Colab session, then rerun cell 1.")

print("Project revision:", actual_revision)
print("Python:", sys.version.split()[0])
print("Benchmark imports ready. Continue to cell 2.")

In [ ]:
#@title 2. Select tasks and mount Drive
from google.colab import drive
TASK_COUNT = 3 #@param [3, 20] {type:"raw"}
RECOVERY_ENABLED = True #@param {type:"boolean"}
if TASK_COUNT not in (3, 20):
    raise ValueError("Select 3 or 20 tasks.")
EXPECTED_PROTOCOL = "judgment_ballot_v3_2"
print("Selected MSCoRe runner:", EXPECTED_PROTOCOL)
drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/blackboard_benchmarks/mscore")
ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
#@title 3. Verify the dataset and start local GPU inference
import hashlib
import json
import urllib.request
from benchmarks.batch import check_readiness
from benchmarks.colab_setup import ensure_service
from benchmarks.preparation import resources
from llm_broker.config import InferenceSettings, build_broker

ready = None
url = "https://huggingface.co/datasets/032564yn/MSCoRe/resolve/796e5e0/Data2/Construction/simple/data.json"
digest = "d1d787c2b9fcc0db6d8f0db373e7bd938114fb86a500170951b4f22684c9d87c"
cache = ROOT / "construction-simple.json"
if cache.exists() and hashlib.sha256(cache.read_bytes()).hexdigest() == digest:
    raw = cache.read_bytes()
else:
    with urllib.request.urlopen(url, timeout=90) as response:
        raw = response.read()
    if hashlib.sha256(raw).hexdigest() != digest:
        raise ValueError("MSCoRe download failed checksum verification.")
    cache.write_bytes(raw)
rows = json.loads(raw)
settings = InferenceSettings(model=resources()["runtime"]["model"], ollama_host="http://127.0.0.1:11434")
service = ensure_service(cpu_smoke=False)
if "broker" in globals():
    broker.close()
broker = build_broker(settings)
ready = check_readiness(broker, settings, cpu_smoke=False)
print("Ready", settings.model, ready["model_digest"])

In [ ]:
#@title 4. Run live discussions and save each result
from datetime import datetime, timezone
from uuid import uuid4
from agents.board_client import InMemoryBoard
from benchmarks.actions import BenchmarkAgent, active_root
from benchmarks.runtime import BenchmarkScheduler, ROSTER
from contracts.schemas import BlackboardStatus
from llm_broker.ledger import usage_scope
from prompts.budget import build_prompt
from scheduler.policies import RoundRobinPolicy

class ReviewScheduler(BenchmarkScheduler):
    def _check_deadlock(self, snapshot):
        # Round robin: opening, two reviews, rebuttal, two further reviews.
        posts = snapshot.contributions
        if len(posts) < 6:
            return False
        if super()._check_deadlock(snapshot):
            return True
        # Two completed rounds with identical individual ballots and an unresolved split.
        width = len(self.registered_agents)
        if len(posts) % width or len(posts) < 2 * width:
            return False
        def ballots(turns):
            return {p.agent_id: p.payload.prediction.claim for p in turns}
        latest = ballots(posts[-width:])
        return (len(latest) == width and len(set(latest.values())) > 1
                and latest == ballots(posts[-2 * width:-width]))

from typing import Literal
from pydantic import BaseModel, Field, ValidationError
from prompts import DecisionParseError

class JudgmentBallot(BaseModel):
    verdict: Literal["是", "否"]
    rationale: str = Field(min_length=1)
    caveat: str
    confidence: float = Field(ge=0, le=1)

class JudgmentAgent(BenchmarkAgent):
    def output_schema(self):
        return JudgmentBallot.model_json_schema()

    def parse_output(self, text):
        try:
            return JudgmentBallot.model_validate_json(text)
        except ValidationError as exc:
            raise DecisionParseError(str(exc)) from exc

    def validate_output(self, ballot, snapshot):
        root = active_root(snapshot)
        explanation = dict(rationale=ballot.rationale, limitations=ballot.caveat)
        action = dict(confidence=ballot.confidence, explanation=explanation)
        if root is None:
            action.update(kind="final", tag="REVISE", output=ballot.verdict)
        elif ballot.verdict == root.payload.prediction.claim:
            action.update(kind="ratify", target_contribution_id=root.contribution_id)
        elif self.agent_id == "analyst":
            action.update(kind="final", tag="REVISE", output=ballot.verdict,
                          target_contribution_id=root.contribution_id)
        else:
            action.update(kind="contribution", tag="REFUTE", target_contribution_id=root.contribution_id,
                          prediction=dict(claim=ballot.verdict, confidence=action.pop("confidence")))
        decision = super().parse_output(json.dumps(action, ensure_ascii=False))
        return super().validate_output(decision, snapshot)

    def read_with_report(self, snapshot, extra_instructions=None, must_keep_ids=(), *, reserved_tokens=0):
        instructions = ("判断命题本身，而不是判断题目是否给出了证明。先找出使原句为假的具体反例；缺少无关细节并不能推翻原句。按原句的字面含义独立判断。verdict 是你对原句的答案（是或否），也是你的正式表决。"
                        "在 rationale 中解释理由。把不改变答案的补充条件放在 caveat 中，没有则留空。"
                        "请勿添加原句没有的限定词。后续讨论时，评估同伴的具体理由，然后重新表决。")
        # The first vote of each agent on the original branch is independent.
        # Recovery and later votes retain the arguments that need reconsideration.
        if not snapshot.initial_context and not any(p.agent_id == self.agent_id for p in snapshot.contributions):
            snapshot = snapshot.model_copy(update={"contributions": [], "active_claim": None})
        budget = self.prompt_token_budget
        return build_prompt(self.persona, snapshot, self.agent_id, self.system_addendum, self.max_history,
                            token_budget=None if budget is None else max(0, budget-reserved_tokens),
                            output_contract=instructions + " " + (extra_instructions or ""),
                            next_action="Submit your own judgment ballot as JSON.", empty_board="Independent vote.")

def run_branch(task_id, question, trial_id, context):
    sid = str(uuid4())
    board = InMemoryBoard()
    board.create_session(task_id, question, context, sid)
    scheduler = ReviewScheduler(sid, board, policy=RoundRobinPolicy(), consensus_threshold=3,
                                   deadlock_window=2, answer_type="string_exact")
    roster = list(ROSTER)
    opener = context.get("alternative_proposer")
    if opener:
        roster.sort(key=lambda entry: entry[0] != opener)
    for name, persona in roster:
        agent = JudgmentAgent(name, persona, broker, model=settings.model, seed=0,
                              counterfactual_capable=False, answer_type="string_exact", **settings.agent_kwargs())
        def turn(snapshot, scheduled, agent=agent):
            return agent.action(snapshot, phase="review_closed" if active_root(snapshot) else "submit")
        scheduler.register_agent(name, agent.role, turn)
    status, error = "turn_limit", None
    with usage_scope(trial_id=trial_id, session_id=sid):
        try:
            for _ in range(15):
                post = scheduler.step()
                if post is None:
                    status = "queue_empty"
                    break
                print(f"  [{post.agent_id}] {post.tag.value} {post.payload.prediction.claim}", flush=True)
                state = board.get_snapshot(sid).status
                if state in (BlackboardStatus.CONSENSUS, BlackboardStatus.DEADLOCK):
                    status = state.value.lower()
                    break
        except Exception as exc:
            status, error = "error", f"{type(exc).__name__}: {exc}"
    snapshot = board.get_snapshot(sid)
    root = active_root(snapshot)
    return dict(task_id=task_id, question=question, status=status, error=error,
                answer=root.payload.prediction.claim if root and root.metadata.get("final_submission") else None,
                votes=scheduler.vote_state(snapshot), trace=[p.model_dump(mode="json") for p in snapshot.contributions],
                session_id=sid)

def discuss(task_id, question):
    trial_id = str(uuid4())
    main = run_branch(task_id, question, trial_id, {})
    branches = [main]
    if RECOVERY_ENABLED and main["status"] == "deadlock":
        print("Deadlock detected. Reopening the question for one recovery attempt.", flush=True)
        prior = [{"agent": p["agent_id"], "tag": p["tag"],
                  "claim": p["payload"]["prediction"]["claim"],
                  "reason": p["payload"]["explanation"]["rationale"][:600]} for p in main["trace"][-6:]]
        dissent = next((p for p in reversed(prior) if p["tag"] == "REFUTE" and p["claim"] != main["answer"]), None)
        branches.append(run_branch(task_id, question, trial_id, {
            "alternative_proposer": dissent["agent"] if dissent else None,
            "original_answer": main["answer"], "prior_discussion": prior,
            "latest_analyst_response": next((p["reason"] for p in reversed(prior) if p["agent"] == "analyst"), ""),
            "outstanding_objection": (dissent or prior[-1])["reason"],
            "recovery_instruction": "Compare the prior objections with the literal statement. Identify any added assumption. "
                                    "Reconsider both labels, propose a justified answer, and review it independently."
        }))
    last = branches[-1]
    # Commit a recovery answer only when the new branch earns consensus.
    accepted = last if last["status"] == "consensus" else main
    return dict(task_id=task_id, question=question, answer=accepted["answer"],
                answer_session_id=accepted["session_id"], status=last["status"], error=last["error"],
                recovery_attempted=len(branches)==2, recovery_resolved=len(branches)==2 and last["status"]=="consensus",
                branches=branches, usage=broker.trial_usage(trial_id).model_dump())

def save(path, record):
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)

if globals().get("NOTEBOOK_VERSION") != "3.2.0":
    raise RuntimeError("Run cell 1 of the v3.2.0 notebook first.")
print("MSCoRe notebook v" + NOTEBOOK_VERSION, flush=True)
RUNNER_PROTOCOL = "judgment_ballot_v3_2"
if globals().get("EXPECTED_PROTOCOL") != RUNNER_PROTOCOL:
    raise RuntimeError("Run cell 2 of this notebook before starting the ballot runner.")
print("Executing MSCoRe protocol:", RUNNER_PROTOCOL, "| Recovery:", RECOVERY_ENABLED, flush=True)
if ready is None:
    raise RuntimeError("Complete cell 3 first.")
selected = rows[:TASK_COUNT]
assert len(selected) == TASK_COUNT and all(row["answer"] in ("是", "否") for row in selected)
run = ROOT / "runs" / (datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S") + "-" + uuid4().hex[:8])
run.mkdir(parents=True)
ids = [f"construction-simple-{i:04d}" for i in range(1, TASK_COUNT+1)]
save(run / "manifest.json", dict(notebook_version=NOTEBOOK_VERSION, task_ids=ids, source=url, sha256=digest, project=PROJECT_REVISION,
                                 settings=settings.to_dict(), readiness=ready, seed=0, protocol=RUNNER_PROTOCOL, recovery_enabled=RECOVERY_ENABLED,
                                 recovery="one_deadlock_branch" if RECOVERY_ENABLED else "disabled"))
print("SAVE THIS RUN FOLDER:", run, flush=True)
for task_id, row in zip(ids, selected):
    print(task_id, row["question"], flush=True)
    result = discuss(task_id, row["question"])
    result.update(reference=row["answer"], correct=result["answer"] == row["answer"])
    result["outcome"] = ("correct_consensus" if result["correct"] else "incorrect_consensus") if result["status"] == "consensus" else result["status"]
    save(run / (task_id + ".json"), result)
    print(result["status"], "answer", result["answer"], "correct", result["correct"], result["error"] or "", flush=True)

In [ ]:
#@title 5. Inspect results and optionally download the traces
import shutil
import pandas as pd
from google.colab import files

results = [json.loads(p.read_text()) for p in sorted(run.glob("construction-simple-*.json"))]
total = len(json.loads((run / "manifest.json").read_text())["task_ids"])
print("Saved", len(results), "/", total, "| Correct", sum(r["correct"] for r in results),
      "| Consensus", sum(r["status"] == "consensus" for r in results))
print("Outcomes:", pd.Series([r.get("outcome", r["status"]) for r in results], dtype=str).value_counts().to_dict())
display(pd.DataFrame([{k:r[k] for k in ("task_id","recovery_attempted","recovery_resolved","answer","reference","correct","status","error")} for r in results]))


In [ ]:
#@title 6. Browse all saved tasks and download playback
import html
import json
import shutil
from pathlib import Path
from importlib.resources import files as package_files
from IPython.display import HTML, display
from google.colab import files

SAVED_RUN_FOLDER = "" #@param {type:"string"}
DOWNLOAD_ZIP = False #@param {type:"boolean"}
folder = Path(SAVED_RUN_FOLDER) if SAVED_RUN_FOLDER.strip() else globals().get("run")
if folder is None:
    raise ValueError("Enter the Drive run folder printed by cell 4.")
manifest = json.loads((folder / "manifest.json").read_text())
template = package_files("benchmarks").joinpath("benchmark_viewer.html").read_text()
roles = [("analyst","data_analyst"),("critic","data_engineer"),("reasoner","general_reasoner")]
def scenario(record):
    steps = []
    def step(title, claim, rationale, status="ACTIVE", **details):
        steps.append(dict(turn=f"Event {len(steps)}", narrative=title, claim=claim, rationale=rationale,
                          status=status, tokens=f'{record["usage"]["calls"]} calls across this task',
                          attribution="Controller", tag="IDLE", agentActive=None, sandboxVisible=False,
                          deadlock=status=="DEADLOCK", deadlockMsg="Recorded scheduler deadlock."))
        steps[-1].update(details)
    for i, branch in enumerate(record["branches"]):
        label = "Recovery" if i else "Original"
        leading = record["question"]
        step(label + " branch opened", leading, "Reconsider the prior disagreement." if i else "Assess the statement.")
        for post in branch["trace"]:
            who = post["agent_id"]
            pred = post["payload"]["prediction"]
            exp = post["payload"]["explanation"]
            if post["tag"] == "REVISE":
                leading = pred["claim"]
            index = [name for name, role in roles].index(who)
            flows = [None]*len(roles)
            flows[index] = dict(hypothesis=pred["claim"], deliberation=exp["rationale"], action=post["tag"],
                                evidence="; ".join(exp["evidence"]), conf=str(pred["confidence"]))
            step(label + " · " + who + " · " + post["tag"], leading, exp["rationale"],
                 agentActive=index, attribution=who, tag=post["tag"], thoughtFlows=flows)
        step(label + " branch ended", leading, branch["error"] or branch["status"], branch["status"].upper())
    step("Independent grade", str(record["answer"]), f'Reference: {record["reference"]} · Correct: {record["correct"]}',
         record["status"].upper())
    hardware = manifest["readiness"]["hardware"]["gpus"]
    return dict(mode="recorded_benchmark", title=record["task_id"], task=record["question"],
                sessionId=record["branches"][0]["session_id"], hardware=", ".join(g["name"] for g in hardware),
                agents=[dict(name=n, role=r) for n,r in roles], steps=steps, sandbox=None)

scenarios = {}
for task in manifest["task_ids"]:
    path = folder / (task + ".json")
    if path.exists():
        scenarios[task] = scenario(json.loads(path.read_text()))
if not scenarios:
    raise ValueError("This folder has no saved recovery tasks yet.")
def js_data(value):
    return json.dumps(value, ensure_ascii=True).replace("<", r"\u003c")
page = """<!doctype html><meta charset=utf-8><title>MSCoRe playback</title>
<style>body{margin:0;background:#101827;color:white;font:16px system-ui}label{display:block;padding:8px}
iframe{width:100%;height:90vh;border:0}</style>
<label>Recorded task <select id="tasks"></select></label><iframe title="Recorded discussion" id="view"></iframe>
<script>const traces=TRACES;
const template=TEMPLATE;
const picker=document.getElementById('tasks');
for(const id of Object.keys(traces)) picker.add(new Option(id,id));
function show(){const trace=JSON.stringify(traces[picker.value]).replaceAll('<',String.fromCharCode(92)+'u003c');
document.getElementById('view').srcdoc=template.replace('<head>',()=>'<head><script>window.LIVE_TRACE_DATA='+trace+';<'+ '/script>');}
picker.onchange=show;show();</script>"""
page = page.replace("TRACES",js_data(scenarios),1).replace("TEMPLATE",js_data(template),1)
(folder / "viewer.html").write_text(page, encoding="utf-8")
display(HTML('<iframe title="MSCoRe task playback" style="width:100%;height:850px;border:0" srcdoc="' +
             html.escape(page, quote=True) + '"></iframe>'))
if DOWNLOAD_ZIP:
    files.download(shutil.make_archive(str(Path("/content") / folder.name), "zip", folder))
